# 04 — Collections Essentials

Notebook 03 gave you the verbs. This notebook gives you the data shapes those verbs operate on. By the end you will know the five collections you reach for daily — `List`, `Vector`, `Set`, `Map`, `Array` — what each is good at, and how to pick between them.

The next notebook (05) will then put higher-order operations like `map`, `filter`, and `fold` to work on these shapes. This one is about the shapes themselves.

## The landscape, in one breath

Scala ships two parallel collection hierarchies — **immutable** and **mutable** — under `scala.collection.immutable` and `scala.collection.mutable`. The default import that comes with every Scala program points at the **immutable** side. When you write `List`, `Vector`, `Set`, `Map`, you are getting the immutable version.

```
  scala.collection
      |
      +-- immutable     <- the default; safe to share
      |     +-- List, Vector, Set, Map, Range, ...
      |
      +-- mutable       <- explicit opt-in
            +-- ArrayBuffer, HashMap, HashSet, ...

  Array  <- separate; thin wrapper over a JVM array
```

Immutable does not mean *slow*. The immutable collections share structure under the hood — adding to a `List` does not copy the rest of it; it points at the existing tail. We will see this in the `List` section.

## Why immutable by default

Three reasons immutability is the default in Scala:

- **Safe sharing.** An immutable collection can be passed across threads, across functions, across executors in a Spark cluster, with zero coordination. No locks. No defensive copies.
- **Easier to reason about.** A value that cannot change is one less moving part. The function you read on page 12 produces the same answer whether page 14 ran first or not.
- **Cheap to derive new versions.** Immutable collections share structure, so producing a *new* collection that is slightly different from the old one is usually a small operation, not a full copy.

Spark's entire programming model is built on this assumption. RDDs and DataFrames are immutable; every operation produces a new one. We will revisit this in notebook 15.

## `List` — the cons cell

`List[A]` is Scala's **immutable singly-linked list**. It is built out of two shapes: `Nil` (the empty list) and `::` (a cons cell — one element prepended to another list).

In [ ]:
val xs = List(1, 2, 3, 4)
// xs: List[Int] = List(1, 2, 3, 4)

val empty = List.empty[Int]
val same = Nil                       // also the empty list

val ys = 0 :: xs                     // prepend with ::
// ys: List[Int] = List(0, 1, 2, 3, 4)

xs.head                              // 1
xs.tail                              // List(2, 3, 4)
xs.isEmpty                           // false
xs.length                            // 4

Internal shape of `List(1, 2, 3)`:

```
  1 :: 2 :: 3 :: Nil
  =
  [1, *] -> [2, *] -> [3, *] -> Nil
```

Each cell holds a value and a pointer to the rest. That layout decides what is fast and what is slow:

- **Fast** — prepend (`0 :: xs`) is O(1). `head`, `tail`, `isEmpty` are O(1). Pattern matching on `head :: tail` is the most idiomatic way to walk a list.
- **Slow** — random access by index is O(n). `xs.last` is O(n). Anything that needs to reach the end has to traverse the whole spine.

Reach for `List` when you build front-to-back and consume front-to-back. Use a different shape when you need indexed access or fast append.

## `Vector` — the default `Seq`

`Vector[A]` is a **balanced 32-way tree**. From the outside it behaves like an indexed sequence; under the hood it's a shallow tree that gives effectively constant-time access at every position.

In [ ]:
val v = Vector(10, 20, 30, 40)
// v: Vector[Int] = Vector(10, 20, 30, 40)

v(2)              // 30  — indexed access, ~ O(1) for practical sizes
v.head            // 10
v.last            // 40
v :+ 50           // append:  Vector(10, 20, 30, 40, 50)
0 +: v            // prepend: Vector(0, 10, 20, 30, 40)
v.updated(1, 99)  // Vector(10, 99, 30, 40)  — new vector, old one untouched

What to remember:

- **`+:` prepends, `:+` appends.** The colon always sits next to the collection.
- **Random access, prepend, and append are all effectively O(1).** Vector is the safe default when you don't know exactly what your access pattern will be.
- `updated(i, newValue)` returns a new vector with one position changed. The original is unchanged. The new vector reuses most of the old one internally.

When in doubt between `List` and `Vector`, pick `Vector`. Pick `List` only when you specifically want cons-cell ergonomics for recursion or pattern matching.

## `Set` — unordered, unique

`Set[A]` holds unique elements with no defined iteration order. Membership is the fast operation.

In [ ]:
val s = Set("red", "green", "blue")

s.contains("red")     // true,    O(1)-ish
s("red")              // true,    same thing — Sets are also functions A => Boolean
s + "yellow"          // Set("red", "green", "blue", "yellow")
s - "red"             // Set("green", "blue")
s union Set("blue", "black")    // Set("red", "green", "blue", "black")
s intersect Set("blue", "black") // Set("blue")
s diff Set("blue")    // Set("red", "green")

Worth registering:

- **A `Set[A]` is also a function `A => Boolean`.** `s("red")` is the same as `s.contains("red")`. This makes sets composable with the higher-order functions you saw in notebook 03 — you can pass a set directly where a predicate is expected.
- The iteration order of an immutable `Set` is **not** defined. Don't write code that depends on it.
- `+`, `-`, `union`, `intersect`, and `diff` all return *new* sets. The original is never modified.

## `Map` — key to value

`Map[K, V]` is the immutable hash map. Keys are unique; each maps to one value.

In [ ]:
val ages = Map("alice" -> 30, "bob" -> 25, "cara" -> 41)
// ages: Map[String, Int] = HashMap(alice -> 30, bob -> 25, cara -> 41)

ages("alice")            // 30 — throws NoSuchElementException if absent
ages.get("alice")        // Some(30)
ages.get("dave")         // None
ages.getOrElse("dave", 0) // 0
ages.contains("alice")   // true

ages + ("dave" -> 22)    // new Map with dave added
ages - "bob"             // new Map without bob
ages.updated("alice", 31) // alice now 31

Two patterns to internalise:

- **`key -> value` is sugar for the tuple `(key, value)`.** The arrow notation is a method available on every value; `"alice" -> 30` returns a `Tuple2[String, Int]`. `Map` is just a collection of such tuples.
- **`get` returns `Option[V]`**, not the value directly. This is how Scala forces you to acknowledge missing keys at the type level instead of relying on a nullable return. Notebook 09 covers `Option` in depth.

Iteration over a Map yields `(key, value)` tuples in an unspecified order:

In [ ]:
for (name, age) <- ages do
  println(s"$name is $age")
// alice is 30
// bob is 25
// cara is 41

## `Array` — the JVM primitive

`Array[A]` is a thin wrapper over a Java array. It is **mutable**, **fixed-size**, and lives outside the immutable collections hierarchy. Use it when you need the exact memory layout (a contiguous block of primitives) or when you are talking to a Java API that expects one.

In [ ]:
val a = Array(1, 2, 3, 4)
// a: Array[Int] = Array(1, 2, 3, 4)

a(0)                  // 1     — index with parens, not square brackets
a(0) = 99             // in-place mutation
a.length              // 4     — fixed; you cannot grow an Array

val zeros = new Array[Int](5)  // Array(0, 0, 0, 0, 0) — default values

Three things to register:

- **Indexing uses `(i)`, not `[i]`.** Square brackets in Scala mean type parameters, never indices.
- **Arrays are mutable.** A `val` binds the array; the contents can still change in place. This is the gap notebook 02 hinted at.
- **Arrays cannot grow.** If you need a resizable indexed sequence, use `Vector` for immutable or `ArrayBuffer` (from `scala.collection.mutable`) for mutable.

In practice you will see `Array` mostly in three places: low-level performance code, JVM/Java interop, and Spark internals (Spark stores partition data as primitive arrays for cache efficiency).

## Tuples

Not a collection in the same sense, but worth meeting now because every `Map` you saw above is built on them. A tuple holds a small fixed number of values of possibly different types.

In [ ]:
val pair: (String, Int) = ("alice", 30)
val triple = ("alice", 30, true)

pair._1     // "alice"
pair._2     // 30

// Destructure on the left side of a val:
val (name, age) = pair      // name = "alice", age = 30

Tuples are great for *very small* groupings. When you find yourself reaching for `._4` or naming a tuple field, switch to a **case class** (notebook 07) — it gives you named fields and the same convenience.

## Picking the right one

A cheatsheet you can scan when you're about to declare a new variable:

| You want | Reach for |
|---|---|
| Build front-to-back, walk recursively | `List` |
| Indexed access, mixed prepend/append | `Vector` (default when in doubt) |
| Membership testing, dedup | `Set` |
| Key → value lookup | `Map` |
| Java interop, contiguous primitives | `Array` |
| Mutable resizable buffer | `mutable.ArrayBuffer` |
| Two or three values bundled together briefly | `Tuple` |
| Named fields on a small record | `case class` (notebook 07) |

## The `val` / immutable gap, revisited

Notebook 02 hinted at this; now we have the example. A `val` is a single-assignment binding. It does **not** make what it points to immutable.

In [ ]:
val arr = Array(1, 2, 3)
// arr = Array(4, 5, 6)   // compile error: val cannot be reassigned
arr(0) = 99               // OK: the array's contents changed in place
// arr is now Array(99, 2, 3)

val xs = List(1, 2, 3)
// xs(0) = 99             // compile error: List has no update method
val ys = xs.updated(0, 99) // produces a new List, xs is unchanged

Two layers of guarantee at work:

- `val` controls **rebinding** — can the name point to a different value tomorrow?
- The collection's type controls **mutation** — does the value itself have methods that change its state?

`val` + immutable collection = both layers locked. That is the default you should reach for. `val` + `Array` = the binding is locked but the contents can change underneath you. `var` + immutable = the contents are stable but the binding can flip — useful in tight loops but rarely needed at higher levels.

## Putting it together

A tiny example that uses four collections cooperating:

In [ ]:
val orders = List(
  ("alice", "book",   12.0),
  ("bob",   "pen",     2.5),
  ("alice", "coffee",  4.5),
  ("cara",  "book",   12.0),
)

val customers: Set[String] = orders.map(_._1).toSet
// customers: Set[String] = Set("alice", "bob", "cara")

val firstByCustomer: Map[String, (String, String, Double)] =
  orders.groupBy(_._1).view.mapValues(_.head).toMap
// firstByCustomer: each customer -> their first order tuple

`List` holds the ordered orders, `Set` dedups customer names, `Map` indexes by customer. Each line returns a new collection without modifying the previous. Notebook 05 will go deeper into operations like `map`, `groupBy`, `view`, and `mapValues`.

## What's next

You now have the data shapes. Notebook 05 puts the verbs and the shapes together — the higher-order operations (`map`, `filter`, `fold`, `groupBy`) that turn one collection into another, and the `for ... yield` sugar that desugars to exactly those operations.